In [1]:
import re
import nltk
import string
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string
import re
from nltk.stem import WordNetLemmatizer
import tensorflow as tf
from tensorflow import keras


In [33]:
from keras.preprocessing.text import Tokenizer
from keras_preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, Embedding, LSTM, GRU, Bidirectional

from gensim.models import Word2Vec

from keras.callbacks import EarlyStopping
from keras.models import load_model


In [3]:
df_train = pd.read_csv('train.txt', names=['Text', 'Emotion'], sep=';')
df_val = pd.read_csv('val.txt', names=['Text', 'Emotion'], sep=';')
df_test = pd.read_csv('test.txt', names=['Text', 'Emotion'], sep=';')

In [4]:
df_train['Text']

0                                  i didnt feel humiliated
1        i can go from feeling so hopeless to so damned...
2         im grabbing a minute to post i feel greedy wrong
3        i am ever feeling nostalgic about the fireplac...
4                                     i am feeling grouchy
                               ...                        
15995    i just had a very brief time in the beanbag an...
15996    i am now turning and i feel pathetic that i am...
15997                       i feel strong and good overall
15998    i feel like this was such a rude comment and i...
15999    i know a lot but i feel so stupid because i ca...
Name: Text, Length: 16000, dtype: object

In [5]:
df_train.head

<bound method NDFrame.head of                                                     Text  Emotion
0                                i didnt feel humiliated  sadness
1      i can go from feeling so hopeless to so damned...  sadness
2       im grabbing a minute to post i feel greedy wrong    anger
3      i am ever feeling nostalgic about the fireplac...     love
4                                   i am feeling grouchy    anger
...                                                  ...      ...
15995  i just had a very brief time in the beanbag an...  sadness
15996  i am now turning and i feel pathetic that i am...  sadness
15997                     i feel strong and good overall      joy
15998  i feel like this was such a rude comment and i...    anger
15999  i know a lot but i feel so stupid because i ca...  sadness

[16000 rows x 2 columns]>

# Cleaning the Data

In [6]:
duplicates = df_train[df_train.duplicated(subset='Text', keep=False)]


In [7]:
df_train[df_train['Text'] == df_train.iloc[364]['Text']]

,Text,Emotion
364,i tend to stop breathing when i m feeling stre...,sadness
6563,i tend to stop breathing when i m feeling stre...,anger


In [8]:
df_train.drop_duplicates(subset='Text', keep='first', inplace=True)


In [9]:
df_train.iloc[1112,0]

'i remember feeling the most terrified i had ever felt in my entire life and that its still affecting me now but ive never thought it accounted to trauma'

In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')

def clean_sentence(sentence):
    # Remove special characters and numbers
    sentence = re.sub(r'[^a-zA-Z]', ' ', sentence)
    # Convert to lowercase
    sentence = sentence.lower()
    # Remove URLs
    sentence = re.sub(r'http\S+', '', sentence)
    # Split into words
    sentence = sentence.split()
    # Remove stopwords
    sentence = [word for word in sentence if not word in set(stopwords.words('english'))]
    # Perform lemmatization
    lemmatizer = WordNetLemmatizer()
    sentence = [lemmatizer.lemmatize(word) for word in sentence]
    # Join the words back together
    sentence = ' '.join(sentence)
    return sentence



In [11]:
##Cleaning the whole datasets

df_train['Text'] = df_train['Text'].apply(lambda x: clean_sentence(x))
df_test['Text'] = df_test['Text'].apply(lambda x: clean_sentence(x))
df_val['Text'] = df_val['Text'].apply(lambda x: clean_sentence(x))

In [12]:
df_train_1=df_train

In [13]:
##creating an extra column for emotion mapping

unique_emotions = df_train_1['Emotion'].unique()
print(unique_emotions)
emotion_mapping = {emotion: idx for idx, emotion in enumerate(unique_emotions)}

print(emotion_mapping)
df_train_1['emotions_encoded'] = df_train_1['Emotion'].map(emotion_mapping)


['sadness' 'anger' 'love' 'surprise' 'fear' 'joy']

{'sadness': 0, 'anger': 1, 'love': 2, 'surprise': 3, 'fear': 4, 'joy': 5}


In [14]:
df_train_1

,Text,Emotion,emotions_encoded
0,didnt feel humiliated,sadness,0
1,go feeling hopeless damned hopeful around some...,sadness,0
2,im grabbing minute post feel greedy wrong,anger,1
3,ever feeling nostalgic fireplace know still pr...,love,2
4,feeling grouchy,anger,1
...,...,...,...
15995,brief time beanbag said anna feel like beaten,sadness,0
15996,turning feel pathetic still waiting table subb...,sadness,0
15997,feel strong good overall,joy,5
15998,feel like rude comment im glad,anger,1


In [15]:
df_train_1.to_csv('df_train_1.csv', index=False)

In [16]:
##saving labels and data in seperate csv files

train_lables=df_train_1['emotions_encoded']
train_lables.to_csv("train_labels.csv", index=False)
train_data=df_train_1['Text']
train_data.to_csv("train_data.csv", index=False)


In [17]:
df_val_1=df_val

In [18]:
##Same thing with validation data

emotion_mapping = emotion_mapping

df_val_1['emotions_encoded'] = df_val_1['Emotion'].map(emotion_mapping)
val_labels=df_val_1['emotions_encoded']
val_labels.to_csv("val_labels.csv", index=False)
val_data=df_val_1['Text']
val_data.to_csv("val_data.csv", index=False)

In [19]:
val_data

0                im feeling quite sad sorry ill snap soon
1       feel like still looking blank canvas blank pie...
2                              feel like faithful servant
3                                     feeling cranky blue
4                                   treat feeling festive
                              ...                        
1995    im ssa examination tomorrow morning im quite w...
1996    constantly worry fight nature push limit inner...
1997           feel important share info experience thing
1998    truly feel passionate enough something stay tr...
1999    feel like wanna buy cute make see online even one
Name: Text, Length: 2000, dtype: object

In [20]:

train_lables=df_train_1['emotions_encoded']
train_lables.to_csv("train_labels.csv", index=False)
train_data=df_train_1['Text']
train_data.to_csv("train_data.csv", index=False)


## Tokenizing and Sequencing

In [21]:
sentences = train_data.tolist()

# Tokenize the sentences into words
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
sequences = tokenizer.texts_to_sequences(sentences)

## Basically we have extracted all the sentences and then assigned each word a number and then buit each sentence as a sequece of numbers

## Padding

In [22]:
max_sequence_length = max([len(seq) for seq in sequences])
padded_sequences = pad_sequences(sequences, maxlen=max_sequence_length)
padded_sequences = np.array(padded_sequences)
vocabulary_size = np.max([np.max(padded_sequences[i]) for i in range(padded_sequences.shape[0])]) + 1

## now we have padded the sentences amd the maximum length was 35 the dimensions of the padded sequence became (15969, 35)

## Encoding Validation Data

In [23]:

sentences_val = val_data.tolist()

# Tokenize the sentences into words
tokenizer_val = Tokenizer()
tokenizer_val.fit_on_texts(sentences_val)
sequences_val = tokenizer_val.texts_to_sequences(sentences_val)

max_sequence_length_val = 35
padded_sequences_val = pad_sequences(sequences_val, maxlen=max_sequence_length_val)
padded_sequences_val = np.array(padded_sequences_val)
vocabulary_size_val = np.max([np.max(padded_sequences_val[i]) for i in range(padded_sequences_val.shape[0])]) + 1


## Encoding is Done, Start with Embedding

In [24]:
glove_file = "glove.6B.200d.txt"
glove = {}

with open(glove_file, "r", encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        glove[word] = coefs

## we have created a glove object that contains 200 dimensional feature vector for all 

In [25]:
word_index=tokenizer.word_index
embedding_matrix = np.zeros((vocabulary_size, 200))

## Returns a list of words along with their position in the data set


In [26]:
for i, word in enumerate(word_index):
    embedding_vector = glove.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

## Now we have put in all the words that are there in our voculablurs and assigned then their vector representation

In [27]:
from keras.utils import to_categorical

In [28]:
train_labels=to_categorical(train_lables)
val_labels=to_categorical(val_labels)

## Model Formation and Training

In [29]:
adam = keras.optimizers.Adam(learning_rate=0.005)

model = Sequential()
model.add(Embedding(vocabulary_size, 200, input_length=35, weights=[embedding_matrix], trainable=False))
model.add(Bidirectional(LSTM(256, dropout=0.2,recurrent_dropout=0.2, return_sequences=True)))
model.add(Bidirectional(LSTM(128, dropout=0.2,recurrent_dropout=0.2, return_sequences=True)))
model.add(Bidirectional(LSTM(128, dropout=0.2,recurrent_dropout=0.2)))
model.add(Dense(6, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])
model.summary()

Model: "sequential"

_________________________________________________________________

 Layer (type)                Output Shape              Param #   


 embedding (Embedding)       (None, 35, 200)           2695800   

                                                                 

 bidirectional (Bidirectiona  (None, 35, 512)          935936    

 l)                                                              

                                                                 

 bidirectional_1 (Bidirectio  (None, 35, 256)          656384    

 nal)                                                            

                                                                 

 bidirectional_2 (Bidirectio  (None, 256)              394240    

 nal)                                                            

                                                                 

 dense (Dense)               (None, 6)                 1542      

                                        

In [30]:
callback = EarlyStopping(
    monitor="val_loss",
    patience=4,
    restore_best_weights=True,
)

In [ ]:
history = model.fit(padded_sequences,
                    train_labels,
                    validation_data=(padded_sequences_val, val_labels),
                    verbose=1,
                    batch_size=256,
                    epochs=30,
                    callbacks=[callback]
                   ) 

In [34]:
model=load_model('TextEmotionDetector.h5')

## Predicting

In [61]:
text=('this is very nice')
clean_text=clean_sentence(text)

In [62]:
clean_text

'nice'

In [63]:
text=tokenizer.texts_to_sequences([text])
text=pad_sequences(text, maxlen=35, truncating='pre')

In [64]:
result={'sadness': 0, 'anger': 1, 'love': 2, 'surprise': 3, 'fear': 4, 'joy': 5}

In [ ]:
result = np.argmax(model.predict(text), axis=-1)[0]
proba =  np.max(model.predict(text))
print(f"{result} : {proba}\n\n")